IPL Match Winner Prediction

In [2]:
import pandas as pd
import torch
import random
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
teams = [
    "Mumbai Indians",
    "Chennai Super Kings",
    "Royal Challengers Bangalore",
    "Kolkata Knight Riders",
    "Delhi Capitals",
    "Rajasthan Royals",
    "Punjab Kings",
    "Sunrisers Hyderabad"
]

venues = [
    "Wankhede Stadium",
    "Chepauk Stadium",
    "M Chinnaswamy Stadium",
    "Eden Gardens",
    "Arun Jaitley Stadium",
    "Sawai Mansingh Stadium",
    "IS Bindra Stadium",
    "Rajiv Gandhi Stadium"
]

toss_decisions = ["bat", "field"]

data = []

num_matches = 200

for _ in range(num_matches):
    team1, team2 = random.sample(teams, 2)
    
    toss_winner = random.choice([team1, team2])
    toss_decision = random.choice(toss_decisions)
    
    venue = random.choice(venues)

    winner = random.choice([team1, team2])
    
    data.append([team1, team2, toss_winner, toss_decision, venue, winner])

df = pd.DataFrame(data, columns=[
    "team1", "team2", "toss_winner", "toss_decision", "venue", "winner"
])

df.to_csv("matches.csv", index=False)

print("Dataset created successfully ")

Dataset created successfully 


In [4]:
df = pd.read_csv("matches.csv")
X = df.drop("winner", axis=1)
y = df["winner"]
encoders = {}

In [5]:
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [7]:
class IPLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.out = nn.Linear(16, len(set(y)))

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.out(x)

model = IPLModel()

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.1068
Epoch 10, Loss: 2.0713
Epoch 20, Loss: 2.0287
Epoch 30, Loss: 1.9813
Epoch 40, Loss: 1.9335


In [9]:
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print("\n Accuracy:", round(accuracy * 100, 2), "%")


 Accuracy: 25.0 %


In [10]:
def predict_match(team1, team2, toss_winner, toss_decision, venue):
    input_data = pd.DataFrame([[team1, team2, toss_winner, toss_decision, venue]],
                              columns=X.columns)
    for col in input_data.columns:
        input_data[col] = encoders[col].transform(input_data[col])
    input_data = scaler.transform(input_data)
    input_tensor = torch.tensor(input_data, dtype=torch.float32)
    with torch.no_grad():
        output = model(input_tensor)
        _, pred = torch.max(output, 1)
    return target_encoder.inverse_transform(pred.numpy())[0]

In [11]:
result = predict_match(
    "Mumbai Indians",
    "Chennai Super Kings",
    "Mumbai Indians",
    "bat",
    "Wankhede Stadium"
)
print("\n Predicted Winner:", result)


 Predicted Winner: Chennai Super Kings
